# SYMFLUENCE Tutorial 04d — Rio Diguillin, Chile (Lumped, ERA5, DGA Observations)

## Introduction

This notebook demonstrates SYMFLUENCE's international modeling capabilities using a Chilean Andean basin with cloud-based data from two continents:

- **Streamflow observations**: DGA (Direccion General de Aguas) via CAMELS-CL / PANGAEA — downloaded automatically
- **Meteorological forcing**: ERA5 reanalysis from Copernicus Climate Data Store
- **Models**: Any SYMFLUENCE model — SUMMA, HBV, SAC-SMA, Xinanjiang, HEC-HMS, or TOPMODEL

The **Rio Diguillin en San Lorenzo** (DGA station 8130002) drains a ~206 km² rain-dominated catchment in the Biobio region of south-central Chile. The basin sits in the western Andes foothills with elevations ranging from ~440 m at the gauge to ~2,600 m. The catchment has a Mediterranean rainfall regime with most precipitation falling in austral winter (May-August).

### Switching Models

Change `SELECTED_MODEL` in the configuration cell. Supported: `'HBV'`, `'SACSMA'`, `'XINANJIANG'`, `'HECHMS'`, `'TOPMODEL'`, `'SUMMA'`.

In [ ]:
# Environment verification
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore', message='.*is an EXPERIMENTAL module.*')
warnings.filterwarnings('ignore', message='.*import failed.*')

print(f"Python: {sys.executable}")

try:
    import symfluence
    print(f"SYMFLUENCE {symfluence.__version__}")
except ImportError:
    print("ERROR: SYMFLUENCE not found.")
    sys.exit(1)

## Step 1 — Configuration

Set up a lumped basin model for the Rio Maipo at El Manzano using ERA5 forcing and DGA streamflow observations.

In [ ]:
from pathlib import Path
from symfluence import SYMFLUENCE
from symfluence.core.config.models import SymfluenceConfig
import os

# Derive paths from repo structure
current_dir = Path.cwd()
if '.ipynb_checkpoints' in str(current_dir):
    current_dir = current_dir.parent
    os.chdir(current_dir)

repo_root = current_dir.parent.parent
data_dir = repo_root.parent / 'SYMFLUENCE_data'

print(f"Repo root: {repo_root}")
print(f"Data dir:  {data_dir}")

# ============================================================================
# MODEL SELECTION
# ============================================================================
SELECTED_MODEL = 'HBV'

config = SymfluenceConfig.from_minimal(
    # Domain
    domain_name='Rio_Diguillin_Chile',
    experiment_id='diguillin_run_1',
    SYMFLUENCE_DATA_DIR=str(data_dir),
    SYMFLUENCE_CODE_DIR=str(repo_root),
    TAUDEM_DIR='default',

    # Model
    model=SELECTED_MODEL,

    # Rio Diguillin en San Lorenzo (DGA 8130002)
    # 206 km^2 rain-dominated basin in the Biobio region
    # Lat: -36.9244, Lon: -71.5756
    pour_point_coords='-36.9244/-71.5756',
    bounding_box_coords='-36.70/-71.80/-37.10/-71.30',

    # Lumped basin
    definition_method='lumped',
    discretization='GRUs',
    lumped_watershed_method='TauDEM',

    # Forcing: ERA5
    data_access='cloud',
    forcing_dataset='ERA5',
    dem_source='copernicus',
    download_dem=True,

    # Observations: DGA (auto-downloads from PANGAEA)
    station_id='8130002',
    streamflow_data_provider='DGA',

    # Period: 4 years (2014-2017, within CAMELS-CL coverage ending 2018)
    time_start='2014-01-01 01:00',
    time_end='2017-12-31 23:00',
    spinup_period='2014-01-01, 2014-12-31',
    calibration_period='2015-01-01, 2016-12-31',
    evaluation_period='2017-01-01, 2017-12-31',

    # Calibration
    OPTIMIZATION_METHODS=['iteration'],
    optimization_target='streamflow',
    iterative_optimization_algorithm='DDS',
    optimization_metric='KGE',
    iterations=100,
)

print(f"Model: {SELECTED_MODEL}")

# Initialize
symfluence = SYMFLUENCE(config)
project_dir = symfluence.managers['project'].setup_project()
pour_point_path = symfluence.managers['project'].create_pour_point()
print(f"Project: {project_dir}")

## Step 2 — Domain Definition

Delineate the Rio Maipo watershed using TauDEM and Copernicus DEM.

In [ ]:
# Step 2a — Acquire geospatial attributes
symfluence.managers['data'].acquire_attributes()
print("Attribute acquisition complete")

In [ ]:
# Step 2b — Watershed delineation
watershed_path = symfluence.managers['domain'].define_domain()
print(f"Watershed: {watershed_path}")

# Step 2c — Discretization
hru_path = symfluence.managers['domain'].discretize_domain()
print(f"HRU: {hru_path}")

## Step 3 — Data Acquisition

Download DGA streamflow observations (auto-downloaded from PANGAEA/CAMELS-CL) and ERA5 meteorological forcing.

In [ ]:
# Step 3a — DGA streamflow observations (downloads from PANGAEA automatically)
symfluence.managers['data'].process_observed_data()
print("DGA streamflow data acquired")

In [ ]:
# Step 3b — ERA5 forcing
symfluence.managers['data'].acquire_forcings()
print("ERA5 forcing acquisition complete")

In [ ]:
# Step 3c — Model-agnostic preprocessing
symfluence.managers['data'].run_model_agnostic_preprocessing()
print("Model-agnostic preprocessing complete")

## Step 4 — Model Execution

Preprocess, run, and postprocess the selected model.

In [ ]:
# Step 4a — Model-specific preprocessing
model_name = config.model.hydrological_model
print(f"Preprocessing for {model_name}...")
symfluence.managers['model'].preprocess_models()
print(f"{model_name} preprocessing complete")

In [ ]:
# Step 4b — Model execution + postprocessing
print(f"Running {model_name}...")
symfluence.managers['model'].run_models()
print("Model execution complete")

symfluence.managers['model'].postprocess_results()
print("Post-processing complete")

## Step 5 — Streamflow Evaluation

Compare simulated vs observed streamflow from the DGA network.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

experiment_id = config.domain.experiment_id

# Load results CSV
results_file = project_dir / "results" / f"{experiment_id}_results.csv"
results_df = pd.read_csv(results_file, index_col=0, parse_dates=True)

sim_cols = [c for c in results_df.columns if 'discharge' in c.lower() and 'obs' not in c.lower()]
sim_col = sim_cols[0]
print(f"Simulation: {sim_col}")

# Load observations
obs_path = project_dir / "data" / "observations" / "streamflow" / "preprocessed" / f"{config.domain.name}_streamflow_processed.csv"
obs_df = pd.read_csv(obs_path, parse_dates=['datetime'])
obs_df.set_index('datetime', inplace=True)
obs_daily = obs_df['discharge_cms'].resample('D').mean()

# Align
sim_series = results_df[sim_col].resample('D').mean()
spinup_end = pd.to_datetime(config.domain.spinup_period.split(',')[1].strip())
sim_series = sim_series[sim_series.index > spinup_end]

common_idx = sim_series.index.intersection(obs_daily.index)
obs_valid = obs_daily.loc[common_idx].dropna()
sim_valid = sim_series.loc[obs_valid.index]
mask = ~(np.isnan(obs_valid.values) | np.isnan(sim_valid.values))
obs_clean, sim_clean = obs_valid[mask], sim_valid[mask]

# Metrics
r = obs_clean.corr(sim_clean)
alpha = sim_clean.std() / obs_clean.std()
beta = sim_clean.mean() / obs_clean.mean()
kge = 1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)
nse = 1 - np.sum((obs_clean - sim_clean)**2) / np.sum((obs_clean - obs_clean.mean())**2)
pbias = 100 * (sim_clean.sum() - obs_clean.sum()) / obs_clean.sum()

print(f"\nPerformance ({model_name}, uncalibrated):")
print(f"  KGE:   {kge:.3f}")
print(f"  NSE:   {nse:.3f}")
print(f"  PBIAS: {pbias:.1f}%")
print(f"  r:     {r:.3f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(obs_clean.index, obs_clean.values, 'b-', label='Observed (DGA)', lw=1.2, alpha=0.7)
axes[0, 0].plot(sim_clean.index, sim_clean.values, 'r-', label=f'Simulated ({model_name})', lw=1.2, alpha=0.7)
axes[0, 0].set_ylabel('Discharge (m\u00b3/s)')
axes[0, 0].set_title('Streamflow Time Series')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].text(0.02, 0.95, f'KGE: {kge:.3f}\nNSE: {nse:.3f}\nBias: {pbias:.1f}%',
                transform=axes[0, 0].transAxes, va='top',
                bbox=dict(facecolor='white', alpha=0.8), fontsize=9)

axes[0, 1].scatter(obs_clean, sim_clean, alpha=0.5, s=10)
mx = max(obs_clean.max(), sim_clean.max())
axes[0, 1].plot([0, mx], [0, mx], 'k--', alpha=0.5)
axes[0, 1].set_xlabel('Observed (m\u00b3/s)')
axes[0, 1].set_ylabel('Simulated (m\u00b3/s)')
axes[0, 1].set_title('Observed vs Simulated')
axes[0, 1].grid(True, alpha=0.3)

mo = obs_clean.groupby(obs_clean.index.month).mean()
ms = sim_clean.groupby(sim_clean.index.month).mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
axes[1, 0].plot(mo.index, mo.values, 'b-o', label='Observed', ms=6)
axes[1, 0].plot(ms.index, ms.values, 'r-o', label='Simulated', ms=6)
axes[1, 0].set_xticks(range(1,13))
axes[1, 0].set_xticklabels(months)
axes[1, 0].set_ylabel('Mean Q (m\u00b3/s)')
axes[1, 0].set_title('Seasonal Regime')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

os_ = obs_clean.sort_values(ascending=False)
ss_ = sim_clean.sort_values(ascending=False)
axes[1, 1].semilogy(np.arange(1,len(os_)+1)/len(os_)*100, os_, 'b-', label='Observed', lw=2)
axes[1, 1].semilogy(np.arange(1,len(ss_)+1)/len(ss_)*100, ss_, 'r-', label='Simulated', lw=2)
axes[1, 1].set_xlabel('Exceedance (%)')
axes[1, 1].set_ylabel('Q (m\u00b3/s)')
axes[1, 1].set_title('Flow Duration Curve')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'Rio Diguillin en San Lorenzo \u2014 {model_name} (DGA + ERA5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 6 — Calibration

Calibrate model parameters using DDS to optimize KGE.

In [ ]:
print(f"Calibrating {model_name}...")
print(f"  Algorithm: {config.optimization.algorithm}")
print(f"  Metric: {config.optimization.metric}")
print(f"  Iterations: {config.optimization.iterations}")

results_file = symfluence.managers['optimization'].calibrate_model()
print(f"\nCalibration complete: {results_file}")

In [ ]:
# View calibration results
if results_file and Path(results_file).exists():
    cal_df = pd.read_csv(results_file)
    cal_df['best_score'] = cal_df['score'].cummax()

    print(f"Initial KGE: {cal_df['best_score'].iloc[0]:.4f}")
    print(f"Final KGE:   {cal_df['best_score'].iloc[-1]:.4f}")
    print(f"Improvement: {cal_df['best_score'].iloc[-1] - cal_df['best_score'].iloc[0]:.4f}")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cal_df['iteration'], cal_df['best_score'], 'b-', lw=2)
    ax.set_xlabel('Iteration')
    ax.set_ylabel(f'Best {config.optimization.metric}')
    ax.set_title(f'{model_name} Calibration Progress — Rio Maipo')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No calibration results found.")